# Lab 1 — Build the whole harness (EP09 · capstone v1)

From the theory video: the textbook closes chapter 18 with **Listing 18.1** — a complete harness in one file (§18.10)
In this lab we write it one class at a time for our research assistant, then run a real task

**What you'll write**
1. Two data types: `Message` and `ToolDefinition`
2. `ContextManager` — the five-part budget, checked on every new message, and the preflight check
3. `ToolExecutor` — the approval gate, timeout + retry, output truncation
4. `LoopDetector` — a hash of each action in a window of 5
5. `AgentHarness` — the ReAct loop, with a `run_id` on every log line
6. Two real runs: **approve** saving the report, and **reject** it

Two differences from the book: we count tokens with Gemma's tokenizer (not GPT-4o's `tiktoken` — as EP07 taught),
and we add a token counter and one log line that says how long the tools took

The paper library and the tools are **simulated data written for teaching**. The tool calls and the model's answers come from real runs.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.10 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs under 1 US cent) |
| Packages | `openai`, `python-dotenv`, `tokenizers` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "tokenizers==0.23.3"

## 1. Model, provider and logging

**Why `google/gemma-4-26b-a4b-it`** — it can call tools and **ask for several tools in one turn**, which the book's harness runs at once with `asyncio.gather`.
Its tokenizer is open to download too, so we can count tokens on our machine exactly as the model does

**Why the provider is pinned to Makora** — the same provider as EP08, re-tested 2026-11-17: it calls tools and asked for three `read_paper` calls in one turn every time.
It sometimes answers 429 (too many requests), so we let the client retry by itself up to 5 times

**Not free**, but very cheap: USD 0.09 per million input tokens and USD 0.28 per million output tokens — at the end of the lab we add up the real cost
· The `:free` version doesn't work (tested 2026-11-17: 2 out of 3 calls got 429 "rate-limited upstream", and the other answered without calling the tool)

In [ ]:
import asyncio, hashlib, json, logging, os, time
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Optional
from openai import AsyncOpenAI
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
tok = Tokenizer.from_pretrained(MODEL)

logging.basicConfig(level=logging.WARNING, format="%(message)s", force=True)
logger = logging.getLogger("agent_harness")
logger.setLevel(logging.INFO)

## 2. Two data types: `Message` and `ToolDefinition`

- `Message` — everything exchanged with the model, with one of four roles. A tool message carries a `tool_call_id` that says which request it answers (EP08)
- `ToolDefinition` — a tool's ID card: three fields go to the model, the other two (`handler`, `requires_approval`) the model never sees

In [ ]:
class Role(str, Enum):
    SYSTEM = "system"
    USER = "user"
    ASSISTANT = "assistant"
    TOOL = "tool"

@dataclass
class Message:
    role: Role
    content: str
    tool_calls: Optional[list[dict]] = None
    tool_call_id: Optional[str] = None

    def to_api_dict(self) -> dict:
        d = {"role": self.role.value, "content": self.content or None}
        if self.tool_calls:
            d["tool_calls"] = self.tool_calls
        if self.tool_call_id:
            d["tool_call_id"] = self.tool_call_id
        return d

In [ ]:
@dataclass
class ToolDefinition:
    name: str
    description: str
    parameters: dict
    handler: Callable
    requires_approval: bool = False

    def to_api_dict(self) -> dict:
        return {"type": "function",
                "function": {"name": self.name, "description": self.description,
                             "parameters": self.parameters}}

## 3. `ContextManager` — check the budget on every new message

| Part | Share (book) |
|---|---|
| system · memory · tools · history · reserved | 10% · 20% · 10% · 50% · 10% |

- History over budget → drop the oldest message, **but always keep the first one**; and if a dropped message asked for tools, drop the tool results that follow it too
- **Preflight check** before every model call: system + tools + history ≤ context size − the reserved part

In [ ]:
def count_tokens(text: str) -> int:
    return len(tok.encode(text).ids)

class ContextManager:
    BUDGET_FRACTIONS = {"system": 0.10, "memory": 0.20, "tools": 0.10,
                        "history": 0.50, "reserved": 0.10}

    def __init__(self, max_tokens: int):
        self.max_tokens = max_tokens
        self.history: list[Message] = []
        self.system_msg: Optional[Message] = None

    def count_message_tokens(self, msg: Message) -> int:
        return count_tokens(msg.content or "") + 4

    def total_history_tokens(self) -> int:
        return sum(self.count_message_tokens(m) for m in self.history)

    def history_budget(self) -> int:
        return int(self.max_tokens * self.BUDGET_FRACTIONS["history"])

    def add_message(self, msg: Message) -> None:
        self.history.append(msg)
        self._enforce_budget()

    def _enforce_budget(self) -> None:
        while self.total_history_tokens() > self.history_budget() and len(self.history) > 2:
            dropped = self.history.pop(1)
            if dropped.tool_calls:
                while len(self.history) > 1 and self.history[1].role == Role.TOOL:
                    self.history.pop(1)

    def preflight_check(self, tool_tokens: int) -> bool:
        sys_tokens = self.count_message_tokens(self.system_msg) if self.system_msg else 0
        total = sys_tokens + tool_tokens + self.total_history_tokens()
        limit = self.max_tokens - int(self.max_tokens * self.BUDGET_FRACTIONS["reserved"])
        if total > limit:
            logger.warning("Context overflow: %d > %d", total, limit)
        return total <= limit

    def build_messages(self) -> list[dict]:
        msgs = [self.system_msg.to_api_dict()] if self.system_msg else []
        return msgs + [m.to_api_dict() for m in self.history]

## 4. `ToolExecutor` — run tools without trusting them

Four steps: ① an unknown tool → an error · ② needs approval → ask a person · ③ timeout + retry (wait 1 → 2 seconds, at most 3 tries) · ④ cut output over 2,000 tokens

**Every way out is a text message back to the model** — none of them crashes the program

In [ ]:
class ToolExecutor:
    MAX_OUTPUT_TOKENS = 2000
    MAX_RETRIES = 3
    TIMEOUT = 30.0

    def __init__(self, tools: list[ToolDefinition], approval_callback: Optional[Callable] = None):
        self.tools = {t.name: t for t in tools}
        self.approval = approval_callback

    async def execute(self, tool_name: str, args: dict) -> str:
        tool = self.tools.get(tool_name)
        if not tool:
            return f"Error: unknown tool '{tool_name}'"
        if tool.requires_approval and self.approval:
            if not await self.approval(tool_name, args):
                return "Action rejected by human reviewer."
        for attempt in range(self.MAX_RETRIES):
            try:
                result = await asyncio.wait_for(self._call(tool, args), timeout=self.TIMEOUT)
                return self._truncate(result)
            except asyncio.TimeoutError:
                logger.warning("Tool %s timed out (attempt %d)", tool_name, attempt + 1)
                if attempt == self.MAX_RETRIES - 1:
                    return f"Error: tool '{tool_name}' timed out"
                await asyncio.sleep(2 ** attempt)
            except Exception as exc:
                logger.error("Tool %s error: %s", tool_name, exc)
                if attempt == self.MAX_RETRIES - 1:
                    return f"Error: {exc}"
                await asyncio.sleep(2 ** attempt)
        return "Error: max retries exceeded"

    async def _call(self, tool: ToolDefinition, args: dict) -> str:
        if asyncio.iscoroutinefunction(tool.handler):
            return str(await tool.handler(**args))
        loop = asyncio.get_running_loop()
        return str(await loop.run_in_executor(None, lambda: tool.handler(**args)))

    def _truncate(self, text: str) -> str:
        enc = tok.encode(text)
        if len(enc.ids) <= self.MAX_OUTPUT_TOKENS:
            return text
        end = enc.offsets[self.MAX_OUTPUT_TOKENS - 1][1]
        return text[:end] + "\n[... output truncated ...]"

## 5. `LoopDetector` — a fingerprint per action

Hash `tool name + arguments` → look at the last 5 → the same one twice = a loop

In [ ]:
class LoopDetector:
    def __init__(self, window: int = 5, max_repeats: int = 2):
        self.window = window
        self.max_repeats = max_repeats
        self.action_hashes: list[str] = []

    def record(self, tool_name: str, args: dict) -> bool:
        h = hashlib.md5(f"{tool_name}:{json.dumps(args, sort_keys=True)}".encode()).hexdigest()
        self.action_hashes.append(h)
        recent = self.action_hashes[-self.window:]
        if recent.count(h) >= self.max_repeats:
            logger.warning("Loop detected: %s called %d times", tool_name, recent.count(h))
            return True
        return False

## 6. `AgentHarness` — the whole ReAct loop

Each round: preflight → call the model → store the reply → no more tool calls? return the answer → run the tools at once → check for a loop → add the results

Three special exits (context full · loop · 50 rounds) each end with a message telling the user what happened · every log line has a `run_id`

In [ ]:
class AgentHarness:
    MAX_ITERATIONS = 50

    def __init__(self, system_prompt: str, tools: list[ToolDefinition],
                 max_tokens: int = 256_000, approval_cb: Optional[Callable] = None):
        self.ctx_mgr = ContextManager(max_tokens)
        self.executor = ToolExecutor(tools, approval_cb)
        self.loop_det = LoopDetector()
        self.tools = tools
        self.ctx_mgr.system_msg = Message(Role.SYSTEM, system_prompt)
        self.usage = {"in": 0, "out": 0}

    async def run(self, user_input: str) -> str:
        run_id = hashlib.md5(f"{time.time()}:{user_input}".encode()).hexdigest()[:8]
        start_ts = time.monotonic()
        logger.info("[%s] Starting run: %s", run_id, user_input[:80])
        self.ctx_mgr.add_message(Message(Role.USER, user_input))
        tool_defs = [t.to_api_dict() for t in self.tools]
        tool_tokens = sum(count_tokens(json.dumps(t)) for t in tool_defs)
        msg = None

        for iteration in range(self.MAX_ITERATIONS):
            if not self.ctx_mgr.preflight_check(tool_tokens):
                logger.error("[%s] Context overflow at iter %d", run_id, iteration)
                return "I've run out of context space. Please start a new conversation."
            try:
                response = await client.chat.completions.create(
                    model=MODEL, messages=self.ctx_mgr.build_messages(),
                    tools=tool_defs or None, tool_choice="auto", temperature=0.0, extra_body=EXTRA)
            except Exception as exc:
                logger.error("[%s] LLM call failed: %s", run_id, exc)
                return f"I encountered an error: {exc}"
            self.usage["in"] += response.usage.prompt_tokens
            self.usage["out"] += response.usage.completion_tokens
            msg, finish = response.choices[0].message, response.choices[0].finish_reason
            self.ctx_mgr.add_message(Message(Role.ASSISTANT, msg.content or "",
                tool_calls=[tc.model_dump() for tc in msg.tool_calls] if msg.tool_calls else None))
            if finish == "stop" or not msg.tool_calls:
                logger.info("[%s] Done in %d iters, %.2fs", run_id, iteration + 1, time.monotonic() - start_ts)
                return msg.content or "Task complete."

            results = await self._execute_tool_calls(msg.tool_calls, run_id)
            for tc in msg.tool_calls:
                if self.loop_det.record(tc.function.name, json.loads(tc.function.arguments)):
                    return "I seem to be stuck in a loop. Please clarify your request."
            for tool_call_id, result in results.items():
                self.ctx_mgr.add_message(Message(Role.TOOL, result, tool_call_id=tool_call_id))

        logger.warning("[%s] Max iterations reached", run_id)
        return ("I reached the maximum number of steps without completing the task. "
                "Here is what I found so far: " + (msg.content or ""))

    async def _execute_tool_calls(self, tool_calls: list, run_id: str) -> dict[str, str]:
        tasks = {}
        for tc in tool_calls:
            try:
                args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                args = {}
            logger.info("[%s] Tool call: %s(%s)", run_id, tc.function.name, args)
            tasks[tc.id] = self.executor.execute(tc.function.name, args)
        t0 = time.monotonic()
        results = await asyncio.gather(*tasks.values(), return_exceptions=True)
        logger.info("[%s] %d tool(s) done in %.1fs", run_id, len(tasks), time.monotonic() - t0)
        return {tid: (f"Error: {r}" if isinstance(r, Exception) else r)
                for tid, r in zip(tasks.keys(), results)}

## 7. The research assistant's three tools

| Tool | What it does | Needs approval |
|---|---|---|
| `search_papers` | searches the simulated library, returns ids + titles (waits 0.5 s) | no |
| `read_paper` | reads one abstract by id (waits 1.5 s, like a slow API) | no |
| `save_report` | saves a report (simulated: kept in a dict, no real file is written) | **yes** 🔒 |

In [ ]:
LIBRARY = {
    "P1": ("Informer (2021)", "ProbSparse self-attention makes long-sequence time-series forecasting faster."),
    "P2": ("PatchTST (2023)", "Splits each series into patches of 16 steps; each channel is handled on its own."),
    "P3": ("TimesFM (2024)", "A decoder-only foundation model that forecasts new series zero-shot."),
    "P4": ("Autoformer (2021)", "Replaces attention with auto-correlation and decomposes trend and season."),
    "P5": ("DLinear (2023)", "A single linear layer that beats many transformers on long-term forecasting."),
    "P6": ("BERT (2019)", "Pre-trains a bidirectional transformer on text with masked language modelling."),
}
SAVED = {}  # the simulated disk

In [ ]:
async def search_papers(query: str, max_results: int = 3) -> str:
    await asyncio.sleep(0.5)
    words = [w.rstrip("s") for w in query.lower().replace("-", " ").split() if len(w) > 3]
    hits = [(pid, t) for pid, (t, a) in LIBRARY.items() if any(w in (t + " " + a).lower() for w in words)]
    return json.dumps([{"id": pid, "title": t} for pid, t in hits[:max_results]])

async def read_paper(paper_id: str) -> str:
    await asyncio.sleep(1.5)
    title, abstract = LIBRARY[paper_id]
    return json.dumps({"id": paper_id, "title": title, "abstract": abstract})

def save_report(filename: str, text: str) -> str:
    SAVED[filename] = text
    return f"saved {len(text)} characters to {filename}"

TOOLS = [
    ToolDefinition("search_papers", "Search the paper library. Returns ids and titles.",
        {"type": "object", "properties": {"query": {"type": "string"},
         "max_results": {"type": "integer", "default": 3}}, "required": ["query"]}, search_papers),
    ToolDefinition("read_paper", "Read one paper's abstract by its id (P1, P2, ...).",
        {"type": "object", "properties": {"paper_id": {"type": "string"}}, "required": ["paper_id"]}, read_paper),
    ToolDefinition("save_report", "Save a finished report to a file.",
        {"type": "object", "properties": {"filename": {"type": "string"}, "text": {"type": "string"}},
         "required": ["filename", "text"]}, save_report, requires_approval=True),
]

## 8. A real run — run 1: approve ✅

In a real app the approval callback asks a person (e.g. `input()` or a button on a web page). In this lab we pass the answer in advance, so the run can be repeated

In [ ]:
SYSTEM = ("You are a research assistant for a university course. Use the tools to find and read papers "
          "before answering. When you need several papers, request them all in the same turn. "
          "Always cite paper ids like [P2].")
TASK = ("Find papers on transformers for time-series forecasting, read the three most relevant ones, "
        "and save a short report to report.md.")

def make_approver(decision: bool):
    async def approve(tool_name: str, args: dict) -> bool:
        print(f"🔒 approval needed: {tool_name}(filename={args.get('filename')!r}, "
              f"text={len(args.get('text', ''))} chars) → {'✅ yes' if decision else '❌ no'}")
        return decision
    return approve

harness = AgentHarness(SYSTEM, TOOLS, approval_cb=make_approver(True))
answer = await harness.run(TASK)
print("\nANSWER:", answer)
print("\nsaved:", list(SAVED), "· usage:", harness.usage)

In [ ]:
print(SAVED["report.md"][:600])

## 9. Run 2: reject ❌

The same task, but this time we answer **no** — `save_report` never runs; the model only gets "Action rejected by human reviewer." What does it do next?

In [ ]:
SAVED.clear()
harness = AgentHarness(SYSTEM, TOOLS, approval_cb=make_approver(False))
answer = await harness.run(TASK)
print("\nANSWER:", answer)
print("\nsaved:", list(SAVED), "· usage:", harness.usage)

In [ ]:
cost = harness.usage["in"] / 1e6 * 0.09 + harness.usage["out"] / 1e6 * 0.28
print(f"run 2: {harness.usage['in']:,} input + {harness.usage['out']:,} output tokens ≈ ${cost:.5f}")

## Summary

| Harness piece | What we saw in the lab |
|---|---|
| `ContextManager` | budget checked on every `add_message` · a dropped tool request takes its results with it · a preflight before every model call |
| `ToolExecutor` | `save_report` needs approval: approved → saved · rejected → the tool never runs |
| Parallel tool calls | three `read_paper` calls in one turn, finished together in about 1.5 seconds |
| `LoopDetector` | in some runs the model asked for `save_report` again after the rejection → the same fingerprint → stop |
| `run_id` | every log line of one run has the same id |

**Exercises**
1. Remove "When you need several papers, request them all in the same turn." from `SYSTEM` and run part 8 again — how many rounds and seconds does it take?
2. Make the model ask for `read_paper("P9")` (not in the library) — how many times does `ToolExecutor` retry, and how long does it wait? Should it retry this kind of error at all?

**Next — Lab 2:** test this harness without calling the model even once, using a mock LLM, unit tests and broken inputs we feed it on purpose